# ⚖️ 검색 가중치 & 스코어 튜닝

Azure AI Search에서 검색 결과의 순위를 조절하는 **모든 가중치 방법**을 학습합니다.

## 📋 학습 내용

| 파트 | 검색 유형 | 가중치 방법 | 특징 |
|------|----------|------------|------|
| Part 1 | 벡터 검색 | `VectorizedQuery(weight=...)` | 쿼리 시점, 벡터 필드 간 비중 |
| Part 2 | 하이브리드 검색 | `VectorizedQuery(weight=...)` | 쿼리 시점, 키워드 vs 벡터 비중 |
| Part 3 | 키워드 검색 | Scoring Profile | **인덱스 정의** 필요, 비즈니스 로직 |

## 🔍 핵심 개념

```
벡터/하이브리드: weight 파라미터로 쿼리마다 자유롭게 조절
키워드(BM25):    Scoring Profile을 인덱스에 정의해야 함
```

---

---
## 1️⃣ 환경 설정 및 초기화

In [1]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    ScoringProfile,
    TextWeights,
    MagnitudeScoringFunction,
    MagnitudeScoringParameters,
    TagScoringFunction,
    TagScoringParameters,
    ScoringFunctionInterpolation,
)
from openai import AzureOpenAI

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_ADMIN_KEY = os.getenv("SEARCH_ADMIN_KEY")
INDEX_NAME = os.getenv("SEARCH_INDEX_NAME", "products-index")
credential = AzureKeyCredential(SEARCH_ADMIN_KEY)

# Azure OpenAI 설정 (벡터 검색용)
openai_endpoint = os.getenv("AZURE_OPEN_AI_ENDPOINT")
openai_key = os.getenv("AZURE_OPEN_AI_KEY")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_EMBEDDING_API_VERSION")

# 클라이언트 초기화
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential)
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
openai_client = AzureOpenAI(azure_endpoint=openai_endpoint, api_key=openai_key, api_version=api_version)

def get_embedding(text):
    response = openai_client.embeddings.create(input=text, model=embedding_deployment)
    return response.data[0].embedding

print(f"✅ 클라이언트 초기화 완료")
print(f"   Endpoint: {SEARCH_ENDPOINT}")
print(f"   Index: {INDEX_NAME}")

✅ 클라이언트 초기화 완료
   Endpoint: https://ai-search-foundry-iq-cj.search.windows.net
   Index: products-index


---

## Part 1: 벡터 검색 가중치

### 멀티 벡터 필드 간 비중 조절

순수 벡터 검색(`search_text=None`)에서 **여러 벡터 필드**를 사용할 때, 각 쿼리의 `weight`로 필드 간 비중을 조절합니다.

> 💡 **동작 원리**: 멀티 벡터 쿼리도 내부적으로 **RRF(Reciprocal Rank Fusion)**로 결합됩니다.  
> `weight`는 RRF 스코어 계산 시 **가중 곱셈 인자**로 적용되어, 해당 벡터 쿼리의 순위 기여도를 높이거나 낮춥니다.

```python
# content_vector (제품 정보) vs review_vector (리뷰)
content_query = VectorizedQuery(vector=..., fields="content_vector", weight=0.5)  # 비중 낮게
review_query  = VectorizedQuery(vector=..., fields="review_vector",  weight=2.0)  # 비중 높게
```

| 시나리오 | content weight | review weight | 이유 |
|----------|---------------|---------------|------|
| 제품 정보 우선 | **2.0** | 0.5 | 브랜드/카테고리 중심 검색 |
| 리뷰 경험 우선 | 0.5 | **2.0** | "배송 빠른", "품질 좋은" 등 |
| 균형 | 1.0 | 1.0 | 기본 설정 |

In [6]:
query_text = "피부에 좋은 에센스"
query_vector = get_embedding(query_text)

print(f"\n{'='*80}")
print(f"⚖️ Part 1: 벡터 검색 - 멀티 벡터 필드 간 가중치 조절")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\ncontent_vector(제품정보)와 review_vector(리뷰)의 비중을 바꿔봅니다.\n")

scenarios = [
    ("제품 정보 우선", 2.0, 0.5),
    ("균형", 1.0, 1.0),
    ("리뷰 경험 우선", 0.5, 2.0),
]

for label, cw, rw in scenarios:
    content_query = VectorizedQuery(
        vector=query_vector, k_nearest_neighbors=10,
        fields="content_vector", weight=cw
    )
    review_query = VectorizedQuery(
        vector=query_vector, k_nearest_neighbors=10,
        fields="review_vector", weight=rw
    )

    results = search_client.search(
        search_text=None,  # 순수 벡터 검색
        vector_queries=[content_query, review_query],
        select=["title", "brand", "category", "normal_price","review"],
        top=3
    )

    print(f"\n{'='*80}")
    print(f"⚖️ {label} (content={cw}, review={rw})")
    print('='*80)
    for idx, r in enumerate(results, 1):
        score = r.get('@search.score', 0)
        print(f"  {idx}. [점수: {score:.4f}] {r['title']}")
        print(f"     {r['brand']} | {r['category']} | {r['normal_price']:,}원")
        print(f"     {r['review']}")

print(f"\n💡 제품 정보 우선 → 제품명/브랜드가 '에센스'에 가까운 제품 상위")
print(f"💡 리뷰 경험 우선 → 리뷰에 '피부에 좋은' 언급이 많은 제품 상위")


⚖️ Part 1: 벡터 검색 - 멀티 벡터 필드 간 가중치 조절
🔍 검색어: '피부에 좋은 에센스'

content_vector(제품정보)와 review_vector(리뷰)의 비중을 바꿔봅니다.


⚖️ 제품 정보 우선 (content=2.0, review=0.5)
  1. [점수: 0.0415] 정샘물 에센셜 스킨 누더 롱웨어 쿠션(본품+리필)(물 토너15ml + 물크림 라이트 8ml+물결크림5ml)
     정샘물 | 이미용 | 45,000원
     정샘물 에센셜 스킨 누더 롱웨어 쿠션을 구매했는데 피부에 자연스럽게 밀착되어 하루 종일 무너짐 없이 깔끔해서 만족스러워요. 특히 본품과 리필뿐만 아니라 토너, 라이트 크림, 물결 크림 샘플이 함께 와서 건조함 없이 촉촉하게 마무리되더라고요. 가벼운 사용감인데도 커버력이 좋아 데일리로 쓰기 좋습니다. 가격대비 구성도 알차고 앞으로도 재구매 의사 있습니다.
  2. [점수: 0.0411] 설화수[8월]윤조에센스 6세대 90ml 기획세트
     설화수 | 이미용 | 140,000원
     평소 건조하고 예민한 피부 때문에 고민이 많았는데 설화수 윤조에센스 6세대 사용 후 확실히 피부가 촉촉해지고 탄력이 생겼어요. 90ml 대용량이라 오래 쓸 수 있어 가성비도 좋고, 흡수도 빨라서 아침저녁으로 부담 없이 바르기 좋아요. 기획세트라 추가로 샘플도 받아 다양한 제품을 체험해볼 수 있어 만족스럽습니다. 꾸준히 사용하면 피부톤이 한층 밝아지는 느낌이라 재구매 의사 있습니다.
  3. [점수: 0.0396] NEW 울트라 훼이셜 크림 4.0세대 50ml
     키엘 | 이미용 | 49,000원
     키엘 울트라 훼이셜 크림 4.0세대는 겨울철 건조한 피부에 정말 효과적이에요. 50ml 용량이라 휴대하기도 편하고, 바르면 촉촉함이 오래가서 하루 종일 당김 없이 편안합니다. 특히 끈적임 없이 부드럽게 스며드는 질감이 마음에 들고, 민감한 피부에도 자극 없이 잘 맞았어요. 가격대는 있지만 꾸준히 사용하기 좋은 가성비 

---

## Part 2: 하이브리드 검색 가중치

### 키워드 vs 벡터 비중 조절

하이브리드 검색에서 `weight`는 **RRF 통합 시 벡터 쪽의 순위 기여도**를 조절합니다.

- 텍스트(BM25) 서브쿼리의 weight는 **암묵적으로 1.0** (변경 불가)
- 벡터 쿼리의 `weight`를 조절하여 상대적 비중 변경
- 텍스트 쪽 기여도를 조절하려면 `maxTextRecallSize`(프리뷰)로 BM25 후보 수를 조절할 수도 있음

```python
vector_query = VectorizedQuery(
    vector=query_vector, fields="content_vector",
    weight=2.0  # 벡터 비중 2배 → 의미 검색 우선
)
results = search_client.search(
    search_text="에센스",        # 키워드 (BM25) - 암묵적 weight 1.0
    vector_queries=[vector_query]  # 벡터 - weight 조절 가능
)
```

| weight 값 | 효과 | 사용 시나리오 |
|-----------|------|--------------|
| **0.3** | 키워드 ⬆️⬆️⬆️ / 벡터 ⬇️ | 정확한 제품명, 모델명 검색 |
| **1.0** | 키워드 = 벡터 | 균형 잡힌 하이브리드 (기본값) |
| **3.0** | 키워드 ⬇️ / 벡터 ⬆️⬆️⬆️ | 자연어 질문, 추천 검색 |

In [2]:
query_text = "피부에 좋은 에센스"
query_vector = get_embedding(query_text)

weights = [0.3, 1.0, 3.0]
weight_labels = {
    0.3: "키워드 우선 (weight=0.3) → '에센스' 단어 매칭 중시",
    1.0: "균형 (weight=1.0) → 기본값",
    3.0: "벡터 우선 (weight=3.0) → '피부에 좋은' 의미 중시"
}

print(f"\n{'='*80}")
print(f"⚖️ Part 2: 하이브리드 검색 - 키워드 vs 벡터 비중 조절")
print(f"🔍 검색어: '{query_text}'")
print('='*80)

all_results = {}

for w in weights:
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=10,
        fields="content_vector",
        weight=w
    )

    results = search_client.search(
        search_text=query_text,  # 하이브리드: 키워드 + 벡터
        vector_queries=[vector_query],
        select=["title", "brand", "category", "normal_price"],
        top=5
    )

    result_list = list(results)
    all_results[w] = result_list

    print(f"\n{'='*80}")
    print(f"⚖️ {weight_labels[w]}")
    print('='*80)
    for idx, r in enumerate(result_list, 1):
        score = r.get('@search.score', 0)
        print(f"  {idx}. [점수: {score:.4f}] {r['title']}")
        print(f"     {r['brand']} | {r['category']} | {r['normal_price']:,}원")

# 순위 변화 비교 테이블
print(f"\n{'='*80}")
print("📊 Weight에 따른 순위 변화 요약")
print('='*80)
print(f"{'제품명':<45} {'w=0.3':>8} {'w=1.0':>8} {'w=3.0':>8}")
print("-" * 75)

all_titles = []
for w in weights:
    for r in all_results[w]:
        if r['title'] not in all_titles:
            all_titles.append(r['title'])

for title in all_titles:
    ranks = []
    for w in weights:
        titles_in_w = [r['title'] for r in all_results[w]]
        if title in titles_in_w:
            ranks.append(str(titles_in_w.index(title) + 1) + "위")
        else:
            ranks.append("  -")
    short_title = title[:40] + "..." if len(title) > 40 else title
    print(f"{short_title:<45} {ranks[0]:>8} {ranks[1]:>8} {ranks[2]:>8}")

print(f"\n💡 weight=0.3: '에센스' 키워드가 정확히 포함된 제품이 상위")
print(f"💡 weight=3.0: '피부에 좋은'이라는 의미를 더 강하게 반영")


⚖️ Part 2: 하이브리드 검색 - 키워드 vs 벡터 비중 조절
🔍 검색어: '피부에 좋은 에센스'

⚖️ 키워드 우선 (weight=0.3) → '에센스' 단어 매칭 중시
  1. [점수: 0.0216] 설화수[8월]윤조에센스 6세대 90ml 기획세트
     설화수 | 이미용 | 140,000원
  2. [점수: 0.0212] NEW 울트라 훼이셜 크림 4.0세대 50ml
     키엘 | 이미용 | 49,000원
  3. [점수: 0.0209] 정샘물 에센셜 스킨 누더 롱웨어 쿠션(본품+리필)(물 토너15ml + 물크림 라이트 8ml+물결크림5ml)
     정샘물 | 이미용 | 45,000원
  4. [점수: 0.0199] [클라랑스]맨 스킨&로션 세트
     클라랑스 | 이미용 | 118,000원
  5. [점수: 0.0190] [스위스유스트] 31허브오일 75
     유스트 | 이미용 | 98,000원

⚖️ 균형 (weight=1.0) → 기본값
  1. [점수: 0.0331] 설화수[8월]윤조에센스 6세대 90ml 기획세트
     설화수 | 이미용 | 140,000원
  2. [점수: 0.0325] 정샘물 에센셜 스킨 누더 롱웨어 쿠션(본품+리필)(물 토너15ml + 물크림 라이트 8ml+물결크림5ml)
     정샘물 | 이미용 | 45,000원
  3. [점수: 0.0325] NEW 울트라 훼이셜 크림 4.0세대 50ml
     키엘 | 이미용 | 49,000원
  4. [점수: 0.0303] [클라랑스]맨 스킨&로션 세트
     클라랑스 | 이미용 | 118,000원
  5. [점수: 0.0299] [스위스유스트] 31허브오일 75
     유스트 | 이미용 | 98,000원

⚖️ 벡터 우선 (weight=3.0) → '피부에 좋은' 의미 중시
  1. [점수: 0.0659] 정샘물 에센셜 스킨 누더 롱웨어 쿠션(본품+리필)(물 토너15ml + 물크림 라이트 8ml+물결크림5ml)
     정샘물 | 이미용 | 45,000원

### 📊 Part 2 Evidence: 하이브리드 Weight 효과 분석

아래 코드는 **동일한 검색어**에 대해 weight 변화가 실제 점수와 순위에 미치는 영향을 정량적으로 분석합니다.

| 관찰 포인트 | 설명 |
|------------|------|
| **점수 변화** | weight↑ → 벡터 기여도↑ → 의미적으로 관련된 결과의 점수 상승 |
| **순위 이동** | 키워드 정확 매칭 제품 vs 의미적 유사 제품의 순위 역전 |
| **Top-1 전환** | weight=0.3과 weight=3.0에서 1위 제품이 바뀌는지 확인 |

In [3]:
# ========================================
# Part 2 Evidence: Weight별 점수 & 순위 변화 정량 분석
# ========================================

query_text = "피부에 좋은 에센스"
query_vector = get_embedding(query_text)

weights = [0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0]

print(f"{'='*90}")
print(f"📊 Evidence: 하이브리드 Weight에 따른 점수 변화 추적")
print(f"🔍 검색어: '{query_text}'")
print(f"{'='*90}")

# 각 weight별 결과 수집
weight_results = {}
for w in weights:
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=10,
        fields="content_vector",
        weight=w
    )
    results = list(search_client.search(
        search_text=query_text,
        vector_queries=[vector_query],
        select=["title", "brand", "category", "normal_price"],
        top=5
    ))
    weight_results[w] = results

# ── Evidence 1: Top-1 제품이 weight에 따라 바뀌는지 ──
print(f"\n{'─'*90}")
print("📌 Evidence 1: Weight별 Top-1 제품 변화")
print(f"{'─'*90}")
print(f"{'Weight':<10} {'점수':<12} {'Top-1 제품':<45} {'브랜드'}")
print(f"{'─'*90}")
for w in weights:
    r = weight_results[w][0]
    score = r.get('@search.score', 0)
    title = r['title'][:42] + "..." if len(r['title']) > 42 else r['title']
    print(f"{w:<10} {score:<12.4f} {title:<45} {r['brand']}")

# ── Evidence 2: 특정 제품의 점수 궤적 ──
print(f"\n{'─'*90}")
print("📌 Evidence 2: 개별 제품의 점수 변화 궤적")
print(f"{'─'*90}")

# 모든 weight 결과에서 등장하는 제품 수집
product_scores = {}
for w in weights:
    for rank, r in enumerate(weight_results[w], 1):
        title = r['title']
        if title not in product_scores:
            product_scores[title] = {}
        product_scores[title][w] = {'score': r.get('@search.score', 0), 'rank': rank}

# 가장 자주 등장하는 상위 제품들만 표시
frequent_products = sorted(product_scores.items(), key=lambda x: len(x[1]), reverse=True)[:5]

header = f"{'제품명':<40}" + "".join(f"{'w='+str(w):<12}" for w in weights)
print(header)
print("─" * (40 + 12 * len(weights)))

for title, scores in frequent_products:
    short_title = title[:37] + "..." if len(title) > 37 else title
    row = f"{short_title:<40}"
    for w in weights:
        if w in scores:
            row += f"{scores[w]['score']:<12.4f}"
        else:
            row += f"{'  -':<12}"
    print(row)

# ── Evidence 3: 순위 이동 분석 ──
print(f"\n{'─'*90}")
print("📌 Evidence 3: 순위 이동 분석 (weight=0.1 vs weight=5.0)")
print(f"{'─'*90}")

low_w = 0.1
high_w = 5.0
titles_low = [r['title'] for r in weight_results[low_w]]
titles_high = [r['title'] for r in weight_results[high_w]]

print(f"\n  ◀ 키워드 우선 (weight={low_w})         ▶ 벡터 우선 (weight={high_w})")
print(f"  {'─'*35}    {'─'*35}")
for i in range(max(len(titles_low), len(titles_high))):
    left = (titles_low[i][:32] + "...") if i < len(titles_low) else ""
    right = (titles_high[i][:32] + "...") if i < len(titles_high) else ""
    print(f"  {i+1}. {left:<35}    {i+1}. {right:<35}")

# 순위 변동 요약
moved_up = []
moved_down = []
for title in set(titles_low + titles_high):
    rank_low = (titles_low.index(title) + 1) if title in titles_low else 6
    rank_high = (titles_high.index(title) + 1) if title in titles_high else 6
    change = rank_low - rank_high
    short = title[:35] + "..." if len(title) > 35 else title
    if change > 0:
        moved_up.append((short, change))
    elif change < 0:
        moved_down.append((short, abs(change)))

if moved_up:
    print(f"\n  🔼 벡터 비중 증가 시 순위 상승:")
    for title, change in sorted(moved_up, key=lambda x: -x[1]):
        print(f"     +{change}단계 ↑  {title}")

if moved_down:
    print(f"\n  🔽 벡터 비중 증가 시 순위 하락:")
    for title, change in sorted(moved_down, key=lambda x: -x[1]):
        print(f"     -{change}단계 ↓  {title}")

print(f"\n{'='*90}")
print("✅ 결론: weight 값이 클수록 의미적 유사성(벡터)이 순위에 더 크게 반영됩니다.")
print("   - 낮은 weight → '에센스'라는 키워드가 정확히 포함된 제품 우선")
print("   - 높은 weight → '피부에 좋은'이라는 의미를 잘 캡처한 제품 우선")

📊 Evidence: 하이브리드 Weight에 따른 점수 변화 추적
🔍 검색어: '피부에 좋은 에센스'

──────────────────────────────────────────────────────────────────────────────────────────
📌 Evidence 1: Weight별 Top-1 제품 변화
──────────────────────────────────────────────────────────────────────────────────────────
Weight     점수           Top-1 제품                                      브랜드
──────────────────────────────────────────────────────────────────────────────────────────
0.1        0.0183       설화수[8월]윤조에센스 6세대 90ml 기획세트                    설화수
0.3        0.0216       설화수[8월]윤조에센스 6세대 90ml 기획세트                    설화수
0.5        0.0249       설화수[8월]윤조에센스 6세대 90ml 기획세트                    설화수
1.0        0.0331       설화수[8월]윤조에센스 6세대 90ml 기획세트                    설화수
2.0        0.0495       설화수[8월]윤조에센스 6세대 90ml 기획세트                    설화수
3.0        0.0659       정샘물 에센셜 스킨 누더 롱웨어 쿠션(본품+리필)(물 토너15ml + 물크림... 정샘물
5.0        0.0992       정샘물 에센셜 스킨 누더 롱웨어 쿠션(본품+리필)(물 토너15ml + 물크림... 정샘물

─────────────────────────────────────────

### 🔎 Part 2 Evidence: 하이브리드 결과의 출처 추적

Azure AI Search는 하이브리드 검색 시 각 결과가 **키워드(BM25)에서 왔는지, 벡터에서 왔는지** 직접 알려주지 않습니다.

하지만 **동일한 쿼리**로 3가지 검색을 각각 수행하면 출처를 역추적할 수 있습니다:

| 검색 방식 | 설정 | 의미 |
|----------|------|------|
| **키워드 전용** | `search_text=쿼리, vector_queries=없음` | BM25 단독 결과 |
| **벡터 전용** | `search_text=None, vector_queries=[...]` | 벡터 유사도 단독 결과 |
| **하이브리드** | `search_text=쿼리, vector_queries=[...]` | RRF 통합 결과 |

결과에 **K** = 키워드에서만 등장, **V** = 벡터에서만 등장, **K+V** = 양쪽 모두 등장으로 표시합니다.

In [4]:
# ========================================
# 하이브리드 검색 결과의 출처 추적 (키워드 vs 벡터)
# ========================================

query_text = "피부에 좋은 에센스"
query_vector = get_embedding(query_text)
TOP_N = 10

# ── 1) 키워드 전용 검색 (BM25) ──
keyword_results = list(search_client.search(
    search_text=query_text,
    vector_queries=None,
    select=["title", "brand", "category", "normal_price"],
    top=TOP_N
))

# ── 2) 벡터 전용 검색 ──
vector_query_only = VectorizedQuery(
    vector=query_vector, k_nearest_neighbors=TOP_N, fields="content_vector"
)
vector_results = list(search_client.search(
    search_text=None,
    vector_queries=[vector_query_only],
    select=["title", "brand", "category", "normal_price"],
    top=TOP_N
))

# ── 3) 하이브리드 검색 (키워드 + 벡터) ──
hybrid_vector = VectorizedQuery(
    vector=query_vector, k_nearest_neighbors=TOP_N, fields="content_vector", weight=1.0
)
hybrid_results = list(search_client.search(
    search_text=query_text,
    vector_queries=[hybrid_vector],
    select=["title", "brand", "category", "normal_price"],
    top=TOP_N
))

# 제목 → 순위 매핑
keyword_ranks = {r['title']: i+1 for i, r in enumerate(keyword_results)}
vector_ranks = {r['title']: i+1 for i, r in enumerate(vector_results)}

print(f"{'='*100}")
print(f"🔎 하이브리드 검색 결과 출처 추적")
print(f"🔍 검색어: '{query_text}'  |  Top {TOP_N}")
print(f"{'='*100}")
print(f"\n{'순위':<5} {'출처':<8} {'제품명':<45} {'키워드순위':<12} {'벡터순위':<12} {'하이브리드점수'}")
print(f"{'─'*100}")

for idx, r in enumerate(hybrid_results, 1):
    title = r['title']
    score = r.get('@search.score', 0)
    k_rank = keyword_ranks.get(title)
    v_rank = vector_ranks.get(title)

    # 출처 판별
    if k_rank and v_rank:
        source = "K+V"
        source_icon = "🔵"
    elif k_rank:
        source = "K"
        source_icon = "🟡"
    else:
        source = "V"
        source_icon = "🟢"

    k_display = f"{k_rank}위" if k_rank else "  -"
    v_display = f"{v_rank}위" if v_rank else "  -"
    short_title = title[:42] + "..." if len(title) > 42 else title

    print(f"  {idx:<4} {source_icon} {source:<5} {short_title:<45} {k_display:<12} {v_display:<12} {score:.4f}")

# ── 요약 통계 ──
hybrid_titles = [r['title'] for r in hybrid_results]
both = sum(1 for t in hybrid_titles if t in keyword_ranks and t in vector_ranks)
keyword_only = sum(1 for t in hybrid_titles if t in keyword_ranks and t not in vector_ranks)
vector_only = sum(1 for t in hybrid_titles if t not in keyword_ranks and t in vector_ranks)

print(f"\n{'─'*100}")
print(f"📊 출처 요약 (하이브리드 Top {TOP_N} 기준)")
print(f"{'─'*100}")
print(f"  🔵 K+V (양쪽 모두 등장): {both}개  → 키워드와 의미 모두 관련")
print(f"  🟡 K   (키워드에서만):    {keyword_only}개  → '에센스' 등 키워드 정확 매칭")
print(f"  🟢 V   (벡터에서만):      {vector_only}개  → '피부에 좋은' 의미적 유사")

print(f"\n💡 RRF(Reciprocal Rank Fusion)는 양쪽에서 모두 높은 순위의 결과를 최상위로 올립니다.")
print(f"💡 한쪽에서만 등장해도 해당 서브쿼리에서 순위가 높으면 하이브리드 결과에 포함됩니다.")

🔎 하이브리드 검색 결과 출처 추적
🔍 검색어: '피부에 좋은 에센스'  |  Top 10

순위    출처       제품명                                           키워드순위        벡터순위         하이브리드점수
────────────────────────────────────────────────────────────────────────────────────────────────────
  1    🔵 K+V   설화수[8월]윤조에센스 6세대 90ml 기획세트                    1위           2위           0.0331
  2    🔵 K+V   정샘물 에센셜 스킨 누더 롱웨어 쿠션(본품+리필)(물 토너15ml + 물크림... 4위           1위           0.0325
  3    🔵 K+V   NEW 울트라 훼이셜 크림 4.0세대 50ml                     2위           3위           0.0325
  4    🔵 K+V   [클라랑스]맨 스킨&로션 세트                              6위           8위           0.0303
  5    🟢 V     [스위스유스트] 31허브오일 75                              -          5위           0.0299
  6    🟢 V     [엘리자베스아덴]그린티 데오드란트 스프레이 150ml 트리플 스페셜 세트        -          9위           0.0280
  7    🟢 V     [데코르테] 리포솜 어드밴스드 리페어 세럼 75ml 세트                 -          6위           0.0258
  8    🟢 V     설화수[8월]퍼펙팅 쿠션 에어리 본품15g+리필15g SPF50+            -          4위           0.0237


---

## Part 3: 키워드 검색 가중치 (Scoring Profile)

### Scoring Profile이란?

벡터/하이브리드의 `weight`와 달리, **키워드 검색(BM25)의 가중치**는 인덱스에 **Scoring Profile**을 정의해야 합니다.

> ⚠️ **적용 범위**: Scoring Profile은 키워드, 벡터, 하이브리드, 시맨틱 검색 모두에서 사용할 수 있지만,  
> **nonvector 필드에만 적용**됩니다. 벡터 유사도 점수 자체에는 영향을 주지 않습니다.  
> 하이브리드 검색에서는 키워드(BM25) 서브쿼리의 순위에 영향을 주고, 그 결과가 RRF 통합에 반영됩니다.

### 주요 구성 요소

| 구성 요소 | 설명 | 적용 예시 |
|----------|------|----------|
| **Text Weights** | 필드별 가중치 (`searchable` 필드) | title 매칭 시 3배 점수 |
| **Magnitude** | 숫자 필드 기반 (`filterable` 필드 필요) | 저가 상품 우선 |
| **Tag** | 특정 값 매칭 (`filterable` 필드 필요) | 프리미엄 브랜드 우선 |
| **Freshness** | 날짜 기반 (`DateTimeOffset` 필드) | 최신 상품 우선 |
| **Distance** | 위치 기반 (`GeographyPoint` 필드) | 가까운 매장 우선 |

### 💡 벡터/하이브리드 weight와의 차이

| 비교 항목 | 벡터/하이브리드 weight | Scoring Profile |
|----------|---------------------|-----------------|
| **설정 위치** | 쿼리 시점 (코드에서) | 인덱스 정의 (사전 설정) |
| **변경 방법** | 파라미터만 변경 | 인덱스 업데이트 필요 |
| **적용 대상** | 벡터 RRF 순위 기여도 | nonvector 필드의 BM25 점수 |
| **유연성** | 쿼리마다 자유 변경 | 프로필 선택만 가능 |

### 점수 계산 과정

```
1. BM25 기본 점수 계산
2. Text Weights 적용 (필드별 가중치 곱셈)
3. Functions 적용 (Magnitude, Tag, Freshness, Distance)
4. functionAggregation으로 여러 함수 결과 통합 (sum/average/min/max/firstMatching)
```

In [5]:
# 인덱스 클라이언트 초기화
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

# ========================================
# Scoring Profile 정의
# ========================================

# 프로파일 1: 필드 가중치 (title 매칭 우선)
profile_field_weights = ScoringProfile(
    name="titleBoost",
    text_weights=TextWeights(
        weights={
            "title": 3.0,      # title 필드 매칭 시 3배
            "brand": 2.0,      # brand 필드 매칭 시 2배
            "review": 1.0      # review 필드는 기본
        }
    )
)

# 프로파일 2: 저가 상품 우선 (Magnitude 함수)
profile_low_price = ScoringProfile(
    name="lowPriceFirst",
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=5,  # 부스팅 배수
            parameters=MagnitudeScoringParameters(
                boosting_range_start=0,       # 0원부터
                boosting_range_end=100000,    # 10만원까지
                constant_boost_beyond_range=False  # 범위 밖은 부스팅 감소
            ),
            interpolation=ScoringFunctionInterpolation.LINEAR  # 선형 감소
        )
    ]
)

# 프로파일 3: 고가 상품 우선 (프리미엄 전략)
profile_high_price = ScoringProfile(
    name="highPriceFirst",
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=5,
            parameters=MagnitudeScoringParameters(
                boosting_range_start=100000,   # 10만원부터
                boosting_range_end=500000,     # 50만원까지
                constant_boost_beyond_range=True  # 범위 밖도 최대 부스팅 유지
            ),
            interpolation=ScoringFunctionInterpolation.LINEAR
        )
    ]
)

# 프로파일 4: 프리미엄 브랜드 우선 (Tag 함수)
profile_premium_brand = ScoringProfile(
    name="premiumBrandFirst",
    functions=[
        TagScoringFunction(
            field_name="brand",
            boost=10,  # 매칭 시 10배 부스팅
            parameters=TagScoringParameters(
                tags_parameter="premiumBrands"  # 쿼리 시 전달할 파라미터 이름
            )
        )
    ]
)

# 프로파일 5: 복합 프로파일 (필드 가중치 + 저가 우선)
profile_combined = ScoringProfile(
    name="bestValue",
    text_weights=TextWeights(
        weights={
            "title": 2.0,
            "brand": 1.5
        }
    ),
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=3,
            parameters=MagnitudeScoringParameters(
                boosting_range_start=0,
                boosting_range_end=80000,
                constant_boost_beyond_range=False
            ),
            interpolation=ScoringFunctionInterpolation.QUADRATIC  # 2차 함수 감소
        )
    ],
    function_aggregation="sum"  # 여러 함수 결과 합산
)

print("✅ 5개의 Scoring Profile 정의 완료")
print("   1. titleBoost: 필드 가중치 (title 3배, brand 2배)")
print("   2. lowPriceFirst: 저가 상품 우선")
print("   3. highPriceFirst: 고가 상품 우선")
print("   4. premiumBrandFirst: 프리미엄 브랜드 우선")
print("   5. bestValue: 복합 (필드 가중치 + 저가 우선)")

constant_boost_beyond_range is not a known attribute of class <class 'azure.search.documents.indexes._generated.models._models_py3.MagnitudeScoringParameters'> and will be ignored
constant_boost_beyond_range is not a known attribute of class <class 'azure.search.documents.indexes._generated.models._models_py3.MagnitudeScoringParameters'> and will be ignored
constant_boost_beyond_range is not a known attribute of class <class 'azure.search.documents.indexes._generated.models._models_py3.MagnitudeScoringParameters'> and will be ignored


✅ 5개의 Scoring Profile 정의 완료
   1. titleBoost: 필드 가중치 (title 3배, brand 2배)
   2. lowPriceFirst: 저가 상품 우선
   3. highPriceFirst: 고가 상품 우선
   4. premiumBrandFirst: 프리미엄 브랜드 우선
   5. bestValue: 복합 (필드 가중치 + 저가 우선)


In [8]:
# 기존 인덱스 가져오기
existing_index = index_client.get_index(INDEX_NAME)

print(f"✅ 기존 인덱스 '{INDEX_NAME}' 로드 완료")
print(f"   - 필드 수: {len(existing_index.fields)}")

# 기존 Scoring Profiles에 새 프로파일 추가
existing_profiles = list(existing_index.scoring_profiles) if existing_index.scoring_profiles else []

# 중복 제거 (이미 있으면 교체)
new_profiles = [
    profile_field_weights,
    profile_low_price,
    profile_high_price,
    profile_premium_brand,
    profile_combined
]

for new_profile in new_profiles:
    existing_profiles = [p for p in existing_profiles if p.name != new_profile.name]
    existing_profiles.append(new_profile)

# 인덱스에 Scoring Profiles 추가
existing_index.scoring_profiles = existing_profiles

# 인덱스 업데이트
result = index_client.create_or_update_index(existing_index)

print(f"\n✅ Scoring Profile 추가 완료!")
print(f"   - 총 Scoring Profiles: {len(result.scoring_profiles)}개")
for profile in result.scoring_profiles:
    print(f"     • {profile.name}")

✅ 기존 인덱스 'products-index' 로드 완료
   - 필드 수: 8

✅ Scoring Profile 추가 완료!
   - 총 Scoring Profiles: 5개
     • titleBoost
     • lowPriceFirst
     • highPriceFirst
     • premiumBrandFirst
     • bestValue


---
### 3-2. 기존 데이터 확인

In [6]:
# 검색 클라이언트 초기화
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential
)

# 기존 데이터 확인
result = search_client.search(search_text="*", top=1, include_total_count=True)
doc_count = result.get_count()

print(f"✅ 기존 인덱스 '{INDEX_NAME}'에 {doc_count}개의 문서가 존재합니다.")
print(f"   → 이전 실습(02-keyword_search, 03-vector_search)에서 업로드한 데이터를 사용합니다.")

✅ 기존 인덱스 'products-index'에 247개의 문서가 존재합니다.
   → 이전 실습(02-keyword_search, 03-vector_search)에서 업로드한 데이터를 사용합니다.


---
### 3-3. Scoring Profile 비교 실습

#### 실습 1: 프로파일 미적용 vs 필드 가중치 적용

In [10]:
def search_with_profile(query, profile_name=None, top=5):
    """Scoring Profile을 적용한 검색"""
    results = search_client.search(
        search_text=query,
        scoring_profile=profile_name,
        top=top,
        include_total_count=True
    )
    
    profile_text = profile_name if profile_name else "없음(기본)"
    print(f"\n🔍 검색어: '{query}'")
    print(f"📊 Scoring Profile: {profile_text}")
    print("=" * 80)
    
    results_list = []
    for idx, result in enumerate(results, 1):
        print(f"{idx}. {result['title'][:50]}...")
        print(f"   브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
        print(f"   점수: {result['@search.score']:.4f}")
        results_list.append({
            'title': result['title'],
            'brand': result['brand'],
            'price': result['normal_price'],
            'score': result['@search.score']
        })
    
    return results_list

In [11]:
# 비교 1: "선물" 검색 - 기본 vs titleBoost
print("\n" + "="*80)
print(" 비교 1: 필드 가중치 효과 (titleBoost)")
print("="*80)

# 기본 검색 (프로파일 없음)
results_default = search_with_profile("선물", profile_name=None, top=5)

# titleBoost 프로파일 적용
results_title = search_with_profile("선물", profile_name="titleBoost", top=5)


 비교 1: 필드 가중치 효과 (titleBoost)

🔍 검색어: '선물'
📊 Scoring Profile: 없음(기본)
1. 사브르 비스트로 샤이니 샐러드 선물세트...
   브랜드: 사브르 | 가격: 47,120원
   점수: 5.8588
2. [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02  선물/인형...
   브랜드: 블루독베이비 | 가격: 36,000원
   점수: 5.2385
3. 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)...
   브랜드: 압소바 | 가격: 39,000원
   점수: 4.6017
4. (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출...
   브랜드: 에뜨와 | 가격: 98,000원
   점수: 4.3339
5. 사브르 비스트로 커트러리 빈티지 디너 선물세트...
   브랜드: 사브르 | 가격: 50,160원
   점수: 4.2787

🔍 검색어: '선물'
📊 Scoring Profile: titleBoost
1. [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02  선물/인형...
   브랜드: 블루독베이비 | 가격: 36,000원
   점수: 11.8577
2. 사브르 비스트로 샤이니 샐러드 선물세트...
   브랜드: 사브르 | 가격: 47,120원
   점수: 10.9179
3. (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출...
   브랜드: 에뜨와 | 가격: 98,000원
   점수: 10.0947
4. [노베즈][선물포장] 촉촉3종세트 (로션+워시+크림) 선물추천 스킨케어선물 화장품세트...
   브랜드: 노베즈 | 가격: 81,000원
   점수: 8.7390
5. [멜리멜로] 생일/집들이선물 클래식 스톤 디퓨저 생일 집들이 친구선물...
   브랜드: 멜리멜로 | 가격: 28,000원
   점수: 8.4638


In [19]:
# 점수 비교 시각화
print("\n📈 점수 비교 (기본 vs titleBoost)")
print("-" * 60)
print(f"{'순위':<5} {'기본 점수':<10} {'titleBoost 점수':<17} {'차이'}")
print("-" * 60)
for i, (d, t) in enumerate(zip(results_default, results_title), 1):
    diff = t['score'] - d['score']
    diff_text = f"+{diff:.4f}" if diff > 0 else f"{diff:.4f}"
    print(f"{i:<5} {d['score']:<15.4f} {t['score']:<15.4f} {diff_text}")


📈 점수 비교 (기본 vs titleBoost)
------------------------------------------------------------
순위    기본 점수      titleBoost 점수     차이
------------------------------------------------------------
1     5.8588          11.8577         +5.9989
2     5.2385          10.9179         +5.6793
3     4.6017          10.0947         +5.4930
4     4.3339          8.7390          +4.4051
5     4.2787          8.4638          +4.1851


### 실습 2: 저가 우선 vs 고가 우선 비교

In [20]:
print("\n" + "="*80)
print(" 비교 2: 가격 기반 부스팅 (저가 vs 고가)")
print("="*80)

# 저가 우선
results_low = search_with_profile("자켓", profile_name="lowPriceFirst", top=5)

# 고가 우선
results_high = search_with_profile("자켓", profile_name="highPriceFirst", top=5)


 비교 2: 가격 기반 부스팅 (저가 vs 고가)

🔍 검색어: '자켓'
📊 Scoring Profile: lowPriceFirst
1. 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 65,550원
   점수: 20.8368
2. 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 6.2902
3. 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 5.8492
4. 라코스테우먼 25SS 여성 오버사이즈 컬러데님 셔츠 자켓 BF2446-55G HCZ...
   브랜드: 라코스테 우먼 | 가격: 213,570원
   점수: 3.7855

🔍 검색어: '자켓'
📊 Scoring Profile: highPriceFirst
1. 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 9.8001
2. 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 9.1131
3. 라코스테우먼 25SS 여성 오버사이즈 컬러데님 셔츠 자켓 BF2446-55G HCZ...
   브랜드: 라코스테 우먼 | 가격: 213,570원
   점수: 8.0846
4. 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 65,550원
   점수: 5.7529


In [21]:
# 가격 순서 비교
print("\n💰 가격 비교")
print("-" * 80)
print(f"{'순위':<5} {'lowPriceFirst':<25} {'가격':<15} {'highPriceFirst':<25} {'가격'}")
print("-" * 80)
for i, (low, high) in enumerate(zip(results_low, results_high), 1):
    low_title = low['title'][:20] + "..."
    high_title = high['title'][:20] + "..."
    print(f"{i:<5} {low_title:<25} {low['price']:>10,}원   {high_title:<25} {high['price']:>10,}원")


💰 가격 비교
--------------------------------------------------------------------------------
순위    lowPriceFirst             가격              highPriceFirst            가격
--------------------------------------------------------------------------------
1     노스페이스 NJ3LR05 남성 아이스...       65,550원   노스페이스 NJ2HR11 남성 패커블...      155,800원
2     노스페이스 NJ2HR11 남성 패커블...      155,800원   노스페이스 NJ2HR41 여성 패커블...      155,800원
3     노스페이스 NJ2HR41 여성 패커블...      155,800원   라코스테우먼 25SS 여성 오버사이즈...      213,570원
4     라코스테우먼 25SS 여성 오버사이즈...      213,570원   노스페이스 NJ3LR05 남성 아이스...       65,550원


### 실습 3: Tag 함수를 활용한 프리미엄 브랜드 부스팅

In [22]:
print("\n" + "="*80)
print(" 비교 3: 프리미엄 브랜드 부스팅 (Tag 함수)")
print("="*80)

# Tag 함수 사용을 위한 검색
# premiumBrands 파라미터에 부스팅할 브랜드 목록 전달
premium_brands = ["노스페이스", "라코스테 우먼", "앤더슨벨"]

# 기본 검색
print("\n📌 기본 검색 (프로파일 없음)")
results_no_tag = search_with_profile("자켓", profile_name=None, top=5)

# Tag 함수 적용 검색
print(f"\n📌 프리미엄 브랜드 우선 (premiumBrands: {premium_brands})")
results_with_tag = search_client.search(
    search_text="자켓",
    scoring_profile="premiumBrandFirst",
    scoring_parameters=[f"premiumBrands-{','.join(premium_brands)}"],
    top=5
)

print("\n🔍 검색어: '자켓'")
print("📊 Scoring Profile: premiumBrandFirst")
print(f"🏷️ 부스팅 브랜드: {premium_brands}")
print("=" * 80)

for idx, result in enumerate(results_with_tag, 1):
    is_premium = "⭐" if result['brand'] in premium_brands else "  "
    print(f"{idx}. {is_premium} {result['title'][:45]}...")
    print(f"      브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
    print(f"      점수: {result['@search.score']:.4f}")


 비교 3: 프리미엄 브랜드 부스팅 (Tag 함수)

📌 기본 검색 (프로파일 없음)

🔍 검색어: '자켓'
📊 Scoring Profile: 없음(기본)
1. 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 6.2902
2. 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 5.8492
3. 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 65,550원
   점수: 5.7529
4. 라코스테우먼 25SS 여성 오버사이즈 컬러데님 셔츠 자켓 BF2446-55G HCZ...
   브랜드: 라코스테 우먼 | 가격: 213,570원
   점수: 3.7855

📌 프리미엄 브랜드 우선 (premiumBrands: ['노스페이스', '라코스테 우먼', '앤더슨벨'])

🔍 검색어: '자켓'
📊 Scoring Profile: premiumBrandFirst
🏷️ 부스팅 브랜드: ['노스페이스', '라코스테 우먼', '앤더슨벨']
1. ⭐ 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
      브랜드: 노스페이스 | 가격: 155,800원
      점수: 62.9018
2. ⭐ 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
      브랜드: 노스페이스 | 가격: 155,800원
      점수: 58.4920
3. ⭐ 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
      브랜드: 노스페이스 | 가격: 65,550원
      점수: 57.5285
4. ⭐ 라코스테우먼 25SS 여성 오버사이즈 컬러데님 셔츠 자켓 BF2446-55G HC...
      브랜드: 라코스테 우먼 | 가격: 213,

### 실습 4: 복합 프로파일 (bestValue)

In [23]:
print("\n" + "="*80)
print(" 비교 4: 복합 프로파일 (필드 가중치 + 저가 우선)")
print("="*80)

# 기본 검색
results_basic = search_with_profile("유아용품 선물", profile_name=None, top=5)

# 복합 프로파일 적용
results_best = search_with_profile("유아용품 선물", profile_name="bestValue", top=5)


 비교 4: 복합 프로파일 (필드 가중치 + 저가 우선)

🔍 검색어: '유아용품 선물'
📊 Scoring Profile: 없음(기본)
1. [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카...
   브랜드: 밍크뮤 | 가격: 35,280원
   점수: 8.6405
2. 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건...
   브랜드: 블루독베이비 | 가격: 22,190원
   점수: 7.2922
3. (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출...
   브랜드: 에뜨와 | 가격: 98,000원
   점수: 6.7053
4. 사브르 비스트로 샤이니 샐러드 선물세트...
   브랜드: 사브르 | 가격: 47,120원
   점수: 5.8588
5. (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/...
   브랜드: 에뜨와 | 가격: 38,000원
   점수: 5.7729

🔍 검색어: '유아용품 선물'
📊 Scoring Profile: bestValue
1. [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카...
   브랜드: 밍크뮤 | 가격: 35,280원
   점수: 21.8218
2. (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/...
   브랜드: 에뜨와 | 가격: 38,000원
   점수: 14.3405
3. 사브르 비스트로 샤이니 샐러드 선물세트...
   브랜드: 사브르 | 가격: 47,120원
   점수: 14.2085
4. 선물추천 [밍크뮤HP]26년말띠선물 (CR)코니밍크애착인형 양말SET(35W70-BHB-0...
   브랜드: 밍크뮤 | 가격: 75,200원
   점수: 14.1826
5. 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건.

---
### 3-4. Interpolation 방식 비교

Magnitude 함수에서 점수가 어떻게 감소하는지 보간 방식을 시각화합니다.

In [24]:
import math

# Interpolation 방식별 점수 계산 함수
def calculate_boost(value, start, end, boost, interpolation):
    """부스팅 점수 계산 (Azure AI Search 방식 시뮬레이션)"""
    if value <= start:
        return boost
    elif value >= end:
        return 1  # 기본 점수
    
    # 정규화된 위치 (0~1)
    position = (value - start) / (end - start)
    
    if interpolation == "linear":
        return boost - (boost - 1) * position
    elif interpolation == "quadratic":
        return boost - (boost - 1) * (position ** 2)
    elif interpolation == "logarithmic":
        return boost - (boost - 1) * math.log10(1 + 9 * position)
    elif interpolation == "constant":
        return boost

# 가격 범위에 따른 부스팅 점수 시각화 (텍스트)
print("\n📊 Interpolation 방식별 부스팅 점수 비교")
print("   (가격 범위: 0원 ~ 100,000원, 부스팅 배수: 5)")
print("=" * 75)
print(f"{'가격':<12} {'Linear':<12} {'Quadratic':<12} {'Logarithmic':<12} {'Constant'}")
print("-" * 75)

prices = [0, 20000, 40000, 60000, 80000, 100000, 150000]
for price in prices:
    linear = calculate_boost(price, 0, 100000, 5, "linear")
    quadratic = calculate_boost(price, 0, 100000, 5, "quadratic")
    logarithmic = calculate_boost(price, 0, 100000, 5, "logarithmic")
    constant = calculate_boost(price, 0, 100000, 5, "constant")
    
    print(f"{price:>10,}원 {linear:<12.2f} {quadratic:<12.2f} {logarithmic:<12.2f} {constant:.2f}")


📊 Interpolation 방식별 부스팅 점수 비교
   (가격 범위: 0원 ~ 100,000원, 부스팅 배수: 5)
가격           Linear       Quadratic    Logarithmic  Constant
---------------------------------------------------------------------------
         0원 5.00         5.00         5.00         5.00
    20,000원 4.20         4.84         3.21         5.00
    40,000원 3.40         4.36         2.35         5.00
    60,000원 2.60         3.56         1.78         5.00
    80,000원 1.80         2.44         1.34         5.00
   100,000원 1.00         1.00         1.00         1.00
   150,000원 1.00         1.00         1.00         1.00


### Interpolation 방식 설명

| 방식 | 특징 | 사용 케이스 |
|------|------|------------|
| **Linear** | 균일하게 감소 | 일반적인 가격 부스팅 |
| **Quadratic** | 처음에는 천천히, 끝에서 급격히 감소 | 저가 구간 강조 |
| **Logarithmic** | 처음에 급격히, 끝에서 천천히 감소 | 중간 가격대 강조 |
| **Constant** | 범위 내 동일 부스팅 | 특정 범위 전체 강조 |

---
### 3-5. 실무 시나리오: 카테고리별 맞춤 검색

In [25]:
print("\n" + "="*80)
print(" 실무 시나리오: 카테고리별 맞춤 검색")
print("="*80)

# 시나리오 1: 유아동 - 가성비 상품 우선
print("\n📍 시나리오 1: 유아동 카테고리 - 가성비 상품 추천")
print("   → lowPriceFirst 프로파일 + category 필터")

results = search_client.search(
    search_text="선물",
    filter="category eq '유아동'",
    scoring_profile="lowPriceFirst",
    top=5
)

print("\n🔍 검색어: '선물' (유아동 카테고리)")
print("-" * 60)
for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['title'][:40]}...")
    print(f"   💰 가격: {result['normal_price']:,}원 | 점수: {result['@search.score']:.4f}")


 실무 시나리오: 카테고리별 맞춤 검색

📍 시나리오 1: 유아동 카테고리 - 가성비 상품 추천
   → lowPriceFirst 프로파일 + category 필터

🔍 검색어: '선물' (유아동 카테고리)
------------------------------------------------------------
1. (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S717...
   💰 가격: 98,000원 | 점수: 21.3227
2. [노베즈][선물포장] 촉촉3종세트 (로션+워시+크림) 선물추천 스킨케어선...
   💰 가격: 81,000원 | 점수: 16.2801
3. 선물추천 [밍크뮤HP]26년말띠선물 (CR)코니밍크애착인형 양말SET(3...
   💰 가격: 75,200원 | 점수: 13.0758
4. [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02...
   💰 가격: 36,000원 | 점수: 12.7820
5. [압소바1] 곰고무연사목욕타올+손수건+인형세트 MIAZA-10911 [임...
   💰 가격: 96,000원 | 점수: 12.5074


In [26]:
# 시나리오 2: 스포츠/레져 - 프리미엄 브랜드 우선
print("\n📍 시나리오 2: 스포츠/레져 카테고리 - 프리미엄 브랜드 추천")
print("   → premiumBrandFirst 프로파일 + category 필터")

sports_premium = ["노스페이스", "파타고니아", "젝시믹스"]

results = search_client.search(
    search_text="자켓 바람막이",
    filter="category eq '스포츠/레져'",
    scoring_profile="premiumBrandFirst",
    scoring_parameters=[f"premiumBrands-{','.join(sports_premium)}"],
    top=5
)

print(f"\n🔍 검색어: '자켓 바람막이' (스포츠/레져 카테고리)")
print(f"🏷️ 프리미엄 브랜드: {sports_premium}")
print("-" * 60)
for idx, result in enumerate(results, 1):
    is_premium = "⭐" if result['brand'] in sports_premium else "  "
    print(f"{idx}. {is_premium} {result['title'][:40]}...")
    print(f"      브랜드: {result['brand']} | 점수: {result['@search.score']:.4f}")


📍 시나리오 2: 스포츠/레져 카테고리 - 프리미엄 브랜드 추천
   → premiumBrandFirst 프로파일 + category 필터

🔍 검색어: '자켓 바람막이' (스포츠/레져 카테고리)
🏷️ 프리미엄 브랜드: ['노스페이스', '파타고니아', '젝시믹스']
------------------------------------------------------------
1. ⭐ 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자...
      브랜드: 노스페이스 | 점수: 220.2112
2. ⭐ 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자...
      브랜드: 노스페이스 | 점수: 213.6705
3. ⭐ 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 ...
      브랜드: 노스페이스 | 점수: 210.1245


---

## ✅ 학습 완료!

### 📊 검색 유형별 가중치 요약

| 검색 유형 | 가중치 방법 | 조절 대상 | 설정 위치 |
|----------|------------|----------|----------|
| **벡터** | `VectorizedQuery(weight=...)` | 벡터 필드 간 RRF 기여도 | 쿼리 시점 |
| **하이브리드** | `VectorizedQuery(weight=...)` | 키워드 vs 벡터 RRF 기여도 | 쿼리 시점 |
| **키워드 (BM25)** | Scoring Profile | nonvector 필드의 점수 부스팅 | 인덱스 정의 |

### Part 1 - 벡터 검색 가중치
- `weight` 파라미터로 **벡터 필드 간 RRF 순위 기여도** 조절
- 멀티 벡터 쿼리도 내부적으로 RRF로 결합됨

### Part 2 - 하이브리드 검색 가중치
- `weight` 파라미터로 **키워드 vs 벡터 RRF 기여도** 조절
- 텍스트(BM25) 쪽은 `maxTextRecallSize`로 후보 수 제어 가능 (프리뷰)

### Part 3 - 키워드 검색 가중치 (Scoring Profile)
- **Text Weights**: 필드별 BM25 점수 가중치
- **Magnitude**: 숫자 필드(가격) 기반 부스팅
- **Tag**: 특정 값(브랜드) 매칭 시 부스팅
- Scoring Profile은 **인덱스에 정의** → 쿼리 시 프로파일 선택
- nonvector 필드에만 적용되지만, 하이브리드 검색의 RRF 결과에도 간접 영향

### 💡 실무 가이드

| 비즈니스 목표 | 권장 방법 |
|--------------|----------|
| 의미 검색 강화 | 하이브리드 weight ↑ |
| 리뷰 기반 추천 | 벡터 review_vector weight ↑ |
| 가성비 상품 추천 | Scoring Profile (Magnitude 저가) |
| 프리미엄 브랜드 노출 | Scoring Profile (Tag 함수) |
| 검색 정확도 향상 | Scoring Profile (TextWeights title 강조) |

### ⚠️ 주의사항

- Scoring Profile은 **nonvector 필드에만 적용** (벡터 유사도 점수 자체는 변경 불가)
- 하이브리드 검색에서는 키워드 서브쿼리의 순위에 영향 → RRF 통합 결과에 간접 반영
- 벡터 weight는 쿼리마다 변경 가능하지만, Scoring Profile은 인덱스 업데이트 필요
- Functions에 사용되는 필드는 반드시 `filterable`로 설정해야 함

In [7]:
# (참고) Scoring Profile 제거 방법
# 특정 프로파일을 제거하려면:
# existing_index = index_client.get_index(INDEX_NAME)
# existing_index.scoring_profiles = [p for p in existing_index.scoring_profiles if p.name not in ['titleBoost', 'lowPriceFirst']]
# index_client.create_or_update_index(existing_index)
# print("✅ Scoring Profile 제거 완료")

print("💡 Scoring Profile은 인덱스에 영구적으로 저장됩니다.")
print("💡 제거하려면 위의 주석 코드를 참고하세요.")

💡 Scoring Profile은 인덱스에 영구적으로 저장됩니다.
💡 제거하려면 위의 주석 코드를 참고하세요.


---

## 🧭 다음 단계

| ⬅️ 이전 | 🏠 목차 | ➡️ 다음 |
|:---------|:-------:|--------:|
| [Lab 04: 하이브리드 검색](../04-hybrid_search/01-hybrid_search.ipynb) | [워크숍 홈](../README.md) | [Lab 06-1: 시맨틱 리랭킹](../06-re_ranking/01-semantic_reranking.ipynb) |